# Resolution Sensitivity v2 — MaxMin Thinning

**Difference from v1**: resolution levels are created by **maxmin-order thinning** of a single
finest-resolution irregular map, not by changing the target grid spacing.

- v1: x8 grid uses 8× coarser spacing → different spatial positions per resolution
- v2: simulate once at finest resolution; take every Rth row in maxmin order → same positions, fewer of them

This isolates the **data quantity** effect (more obs → better estimation) from the
**spatial coverage** effect (coarser grid → different positions).

**Design**: 2 (true_smooth) × 4 (thinning R) × 2 (fit_smooth) = 16 fits

In [ ]:
import os, sys, random, time, gc, pickle
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
from pathlib import Path
from sklearn.neighbors import BallTree

# Absolute path — CWD-independent (mirrors what data_loader.py itself uses)
sys.path.insert(0, '/Users/joonwonlee/Documents/GEMS_TCO-1/src')

from GEMS_TCO import configuration as config
from GEMS_TCO import orderings as _orderings
from GEMS_TCO.data_loader import load_data_dynamic_processed
from GEMS_TCO.kernels_vecchia_hybrid import HybridVecchiaFit

print('torch:', torch.__version__)

In [ ]:
# ── grid / region ──────────────────────────────────────────────────────
DELTA_LAT_BASE = 0.044
DELTA_LON_BASE = 0.063
T_STEPS        = 8
LAT_RANGE      = [-3.0,  2.0]
LON_RANGE      = [121.0, 131.0]

# ── GEMS obs pattern ───────────────────────────────────────────────────
YEAR  = '2023'
MONTH = 7

# ── high-res simulation grid (use 100/10 on Amarel) ───────────────────
HR_LAT_FACTOR = 10
HR_LON_FACTOR = 2

# ── thinning factors: every Rth row from maxmin-ordered finest grid ───
# R=8 → fewest obs (coarsest); R=1 → all obs (finest)
RESOLUTION_MULTS = [8, 4, 2, 1]

# ── hybrid Vecchia spec ────────────────────────────────────────────────
MM_COND_NUMBER  = 100
NHEADS          = 0
DAILY_STRIDE    = 2
LIMIT_A         = 20
LIMIT_B_LOCAL   = 16
LIMIT_C_LOCAL   = 12
LAG1_FRESH      = 2
LAG2_FRESH      = 2
LAG1_LON_OFFSET = 0.063

# ── optimizer ─────────────────────────────────────────────────────────
LBFGS_STEPS = 6
LBFGS_EVAL  = 20
LBFGS_HIST  = 10
LBFGS_LR    = 1.0

# ── true parameters ───────────────────────────────────────────────────
TRUE_DICT = {
    'sigmasq':    10.0,
    'range_lat':   0.5,
    'range_lon':   0.6,
    'range_time':  2.5,
    'advec_lat':   0.08,
    'advec_lon':  -0.16,
    'nugget':      1.2,
}
P_LABELS = ['sigmasq', 'range_lat', 'range_lon', 'range_time',
            'advec_lat', 'advec_lon', 'nugget']
P_DISP   = [r'$\sigma^2$', r'$r_{lat}$', r'$r_{lon}$', r'$r_{time}$',
            r'$\alpha_{lat}$', r'$\alpha_{lon}$', 'nugget']

TRUE_SMOOTHS = [0.5, 1.5]
FIT_SMOOTHS  = [0.5, 1.5]

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DTYPE  = torch.float64
SEED   = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print('Device:', DEVICE)

In [ ]:
# ── parameter reparametrization (identical to v1) ──────────────────────
def true_to_log_params(d):
    phi2 = 1.0 / d['range_lon']
    phi1 = d['sigmasq'] * phi2
    phi3 = (d['range_lon'] / d['range_lat']) ** 2
    phi4 = (d['range_lon'] / d['range_time']) ** 2
    return [np.log(phi1), np.log(phi2), np.log(phi3), np.log(phi4),
            d['advec_lat'], d['advec_lon'], np.log(d['nugget'])]


def backmap_params(out_params):
    p    = [float(x.item() if isinstance(x, torch.Tensor) else x) for x in out_params[:7]]
    phi2 = np.exp(p[1])
    phi3 = np.exp(p[2])
    phi4 = np.exp(p[3])
    rlon = 1.0 / phi2
    return {
        'sigmasq':    np.exp(p[0]) / phi2,
        'range_lat':  rlon / phi3 ** 0.5,
        'range_lon':  rlon,
        'range_time': rlon / phi4 ** 0.5,
        'advec_lat':  p[4],
        'advec_lon':  p[5],
        'nugget':     np.exp(p[6]),
    }


true_log    = true_to_log_params(TRUE_DICT)
true_params = torch.tensor(true_log, device=DEVICE, dtype=DTYPE)
init_log    = true_log   # start from true → isolate identifiability, not init sensitivity
print('true_log:', [f'{v:.3f}' for v in true_log])

In [ ]:
# ── covariance + simulation helpers (identical to v1) ─────────────────
def get_covariance_on_grid(lx, ly, lt, params, smooth=0.5):
    """
    d = sqrt(phi3*(lx-advec_lat*lt)^2 + (ly-advec_lon*lt)^2 + phi4*lt^2)
    smooth=0.5: C = (phi1/phi2)*exp(-phi2*d)
    smooth=1.5: C = (phi1/phi2)*(1+phi2*d)*exp(-phi2*d)
    """
    params   = torch.clamp(params, min=-15.0, max=15.0)
    phi1, phi2, phi3, phi4 = (torch.exp(params[i]) for i in range(4))
    u_lat    = lx - params[4] * lt
    u_lon    = ly - params[5] * lt
    d        = torch.sqrt(u_lat.pow(2) * phi3 + u_lon.pow(2) + lt.pow(2) * phi4 + 1e-8)
    scaled_d = d * phi2
    if smooth == 0.5:
        return (phi1 / phi2) * torch.exp(-scaled_d)
    else:
        return (phi1 / phi2) * (1.0 + scaled_d) * torch.exp(-scaled_d)


def generate_field_values(lats_hr, lons_hr, t_steps, params, dlat, dlon, smooth=0.5):
    cpu = torch.device('cpu')
    f32 = torch.float32
    nx, ny, nt = len(lats_hr), len(lons_hr), t_steps
    px, py, pt = 2 * nx, 2 * ny, 2 * nt
    lx = torch.arange(px, device=cpu, dtype=f32) * dlat
    lx[px // 2:] -= px * dlat
    ly = torch.arange(py, device=cpu, dtype=f32) * dlon
    ly[py // 2:] -= py * dlon
    lt = torch.arange(pt, device=cpu, dtype=f32)
    lt[pt // 2:] -= pt
    params_cpu     = params.cpu().float()
    lx_g, ly_g, lt_g = torch.meshgrid(lx, ly, lt, indexing='ij')
    cov  = get_covariance_on_grid(lx_g, ly_g, lt_g, params_cpu, smooth=smooth)
    spec = torch.fft.fftn(cov)
    spec.real = torch.clamp(spec.real, min=0)
    noise = torch.fft.fftn(torch.randn(px, py, pt, device=cpu, dtype=f32))
    field = torch.fft.ifftn(torch.sqrt(spec.real) * noise).real[:nx, :ny, :nt]
    return field.to(dtype=DTYPE, device=DEVICE)


def apply_step3_1to1(src_np_valid, grid_coords_np, grid_tree, dlat_half, dlon_half):
    n_grid = len(grid_coords_np)
    if len(src_np_valid) == 0:
        return np.full(n_grid, -1, dtype=np.int64)
    dist_to_cell, cell_for_obs = grid_tree.query(np.radians(src_np_valid), k=1)
    dist_to_cell = dist_to_cell.flatten()
    cell_for_obs = cell_for_obs.flatten()
    assignment   = np.full(n_grid, -1, dtype=np.int64)
    best_dist    = np.full(n_grid, np.inf)
    for obs_i, (cell_j, dist) in enumerate(zip(cell_for_obs, dist_to_cell)):
        if dist < best_dist[cell_j]:
            assignment[cell_j] = obs_i
            best_dist[cell_j]  = dist
    filled = assignment >= 0
    if filled.any():
        win_obs  = assignment[filled]
        lat_diff = np.abs(src_np_valid[win_obs, 0] - grid_coords_np[filled, 0])
        lon_diff = np.abs(src_np_valid[win_obs, 1] - grid_coords_np[filled, 1])
        too_far  = (lat_diff > dlat_half) | (lon_diff > dlon_half)
        assignment[np.where(filled)[0][too_far]] = -1
    return assignment


def precompute_mapping_indices(ref_day_map, lats_hr, lons_hr,
                               grid_coords, sorted_keys, dlat_half, dlon_half):
    hr_lat_g, hr_lon_g = torch.meshgrid(lats_hr, lons_hr, indexing='ij')
    hr_coords_np  = torch.stack([hr_lat_g.flatten(), hr_lon_g.flatten()], dim=1).cpu().numpy()
    hr_tree       = BallTree(np.radians(hr_coords_np), metric='haversine')
    grid_coords_np = grid_coords.cpu().numpy()
    n_grid         = len(grid_coords_np)
    grid_tree      = BallTree(np.radians(grid_coords_np), metric='haversine')
    s3_list, hr_list, src_list = [], [], []
    for key in sorted_keys:
        df = ref_day_map.get(key)
        if df is None or len(df) == 0:
            s3_list.append(np.full(n_grid, -1, dtype=np.int64))
            hr_list.append(torch.zeros(0, dtype=torch.long, device=DEVICE))
            src_list.append(torch.zeros((0, 2), dtype=DTYPE, device=DEVICE))
            continue
        src_np       = df[['Source_Latitude', 'Source_Longitude']].values
        valid_mask   = ~np.isnan(src_np).any(axis=1)
        src_np_valid = src_np[valid_mask]
        assignment   = apply_step3_1to1(src_np_valid, grid_coords_np, grid_tree,
                                        dlat_half, dlon_half)
        s3_list.append(assignment)
        if len(src_np_valid) > 0:
            _, hi = hr_tree.query(np.radians(src_np_valid), k=1)
            hr_list.append(torch.tensor(hi.flatten(), device=DEVICE))
        else:
            hr_list.append(torch.zeros(0, dtype=torch.long, device=DEVICE))
        src_list.append(torch.tensor(src_np_valid, device=DEVICE, dtype=DTYPE))
    return s3_list, hr_list, src_list


def assemble_irregular_map(field, s3_list, hr_list, src_list,
                           sorted_keys, grid_coords, true_params, t_offset=21.0):
    nugget_std = torch.sqrt(torch.exp(true_params[6]))
    n_grid     = grid_coords.shape[0]
    field_flat = field.reshape(-1, T_STEPS)
    irr_map    = {}
    for t_idx, key in enumerate(sorted_keys):
        t_val    = float(t_offset + t_idx)
        assign   = s3_list[t_idx]
        hr_idx   = hr_list[t_idx]
        src_locs = src_list[t_idx]
        dummy    = torch.zeros(7, device=DEVICE, dtype=DTYPE)
        if t_idx > 0:
            dummy[t_idx - 1] = 1.0
        rows = torch.full((n_grid, 11), float('nan'), device=DEVICE, dtype=DTYPE)
        rows[:, 3]  = t_val
        rows[:, 4:] = dummy.unsqueeze(0).expand(n_grid, -1)
        if hr_idx.shape[0] > 0:
            gp_vals  = field_flat[hr_idx, t_idx]
            sim_vals = gp_vals + torch.randn(hr_idx.shape[0], device=DEVICE, dtype=DTYPE) * nugget_std
            assign_t = torch.tensor(assign, device=DEVICE, dtype=torch.long)
            filled   = assign_t >= 0
            win_obs  = assign_t[filled]
            rows[filled, 0] = src_locs[win_obs, 0]
            rows[filled, 1] = src_locs[win_obs, 1]
            rows[filled, 2] = sim_vals[win_obs]
        irr_map[key] = rows.detach()
    return irr_map


print('Helper functions defined.')

In [ ]:
# ── load GEMS obs pattern (same as v1) ────────────────────────────────
is_amarel   = os.path.exists(config.amarel_data_load_path)
data_path   = config.amarel_data_load_path if is_amarel else config.mac_data_load_path
data_loader = load_data_dynamic_processed(data_path)

df_map, _, _, _ = data_loader.load_maxmin_ordered_data_bymonthyear(
    lat_lon_resolution=[1, 1],
    mm_cond_number=MM_COND_NUMBER,
    years_=[YEAR], months_=[MONTH],
    lat_range=LAT_RANGE, lon_range=LON_RANGE,
    is_whittle=False,
)
print(f'Loaded {len(df_map)} time slots for {YEAR}-{MONTH:02d}')

yr2      = YEAR[2:]
tco_path = Path(data_path) / f'pickle_{YEAR}' / f'tco_grid_{yr2}_{MONTH:02d}.pkl'
with open(tco_path, 'rb') as f:
    tco_map = pickle.load(f)

all_sorted = sorted(df_map.keys())
day_keys   = all_sorted[:T_STEPS]
ref_day    = {k: tco_map.get(k.split('_', 2)[-1], pd.DataFrame()) for k in day_keys}
print(f'Day pattern: {day_keys[0]} … {day_keys[-1]}')

In [ ]:
# ── high-res simulation grid ───────────────────────────────────────────
dlat_hr = DELTA_LAT_BASE / HR_LAT_FACTOR
dlon_hr = DELTA_LON_BASE / HR_LON_FACTOR
lats_hr = torch.arange(LAT_RANGE[0] - 0.1, LAT_RANGE[1] + 0.1, dlat_hr, device=DEVICE, dtype=DTYPE)
lons_hr = torch.arange(LON_RANGE[0] - 0.1, LON_RANGE[1] + 0.1, dlon_hr, device=DEVICE, dtype=DTYPE)
print(f'HR grid: {len(lats_hr)} lat × {len(lons_hr)} lon = {len(lats_hr)*len(lons_hr):,} cells')

In [ ]:
# ── simulate one GP field per true_smooth (same SEED → comparable) ────
fields = {}
for ts in TRUE_SMOOTHS:
    print(f'Simulating smooth={ts}...', end=' ')
    t0 = time.time()
    torch.manual_seed(SEED)
    fields[ts] = generate_field_values(
        lats_hr, lons_hr, T_STEPS, true_params, dlat_hr, dlon_hr, smooth=ts
    )
    print(f'Done in {time.time()-t0:.1f}s | shape={tuple(fields[ts].shape)}')

In [ ]:
# ── build finest-resolution (R=1) grid + maxmin ordering ─────────────
# This is the single shared spatial structure for ALL thinning levels.

dlat_1 = DELTA_LAT_BASE
dlon_1 = DELTA_LON_BASE

lats_g1 = torch.arange(LAT_RANGE[0], LAT_RANGE[1] + 1e-4, dlat_1, device=DEVICE, dtype=DTYPE)
lons_g1 = torch.arange(LON_RANGE[0], LON_RANGE[1] + 1e-4, dlon_1, device=DEVICE, dtype=DTYPE)
lats_g1 = torch.round(lats_g1 * 10000) / 10000
lons_g1 = torch.round(lons_g1 * 10000) / 10000
g_lat, g_lon   = torch.meshgrid(lats_g1, lons_g1, indexing='ij')
grid_coords_1  = torch.stack([g_lat.flatten(), g_lon.flatten()], dim=1)
n_grid_full    = grid_coords_1.shape[0]
print(f'Finest grid (R=1): {len(lats_g1)} lat × {len(lons_g1)} lon = {n_grid_full:,} cells')

# Maxmin ordering on finest grid (lat, lon) — consistent with v1
coords_np_1       = grid_coords_1.cpu().numpy()
ord_grid_1        = _orderings.maxmin_cpp(coords_np_1)
ordered_coords_1  = coords_np_1[ord_grid_1]   # shape (n_grid_full, 2), (lat, lon)
nns_full          = _orderings.find_nns_l2(locs=ordered_coords_1, max_nn=MM_COND_NUMBER)
print(f'MaxMin + NNS done: nns_full shape={nns_full.shape}')

# Obs → finest grid assignment (tolerance = half a base cell)
s3_list_1, hr_list_1, src_list_1 = precompute_mapping_indices(
    ref_day, lats_hr, lons_hr, grid_coords_1,
    day_keys, dlat_half=dlat_1 / 2, dlon_half=dlon_1 / 2,
)
n_obs_full = sum(int(np.sum(a >= 0)) for a in s3_list_1)
print(f'Matched obs at R=1: {n_obs_full} total ({n_obs_full/T_STEPS:.0f}/step avg)')

In [ ]:
# ── assemble full irr_map (R=1) per true_smooth, then thin per R ──────
#
# v1: irr_maps[(ts, R)] came from a separate R-resolution grid each time.
# v2: irr_maps[(ts, R)] = first round(N/R) rows of maxmin-ordered finest map.
#     MaxMin order: front = most spatially spread (low-freq), back = high-freq.
#     R=8 → first ~N/8 rows (low-freq only)
#     R=4 → first ~N/4 rows (low + some mid-freq)
#     R=2 → first ~N/2 rows (low + mid-freq)
#     R=1 → all N rows (all frequencies)

irr_maps = {}

for ts in TRUE_SMOOTHS:
    # Assemble at finest resolution
    irr_full = assemble_irregular_map(
        fields[ts],
        s3_list_1, hr_list_1, src_list_1,
        day_keys, grid_coords_1, true_params,
    )
    # Apply maxmin ordering
    irr_full_ord = {k: v[ord_grid_1] for k, v in irr_full.items()}
    del irr_full; gc.collect()

    # Thin: first round(N/R) rows in maxmin order (progressive low→high freq)
    for R in RESOLUTION_MULTS:
        n_thin = round(n_grid_full / R)
        thin_idx = np.arange(0, n_thin)
        irr_maps[(ts, R)] = {k: v[thin_idx] for k, v in irr_full_ord.items()}

    del irr_full_ord; gc.collect()

print('Irregular maps assembled:')
for ts in TRUE_SMOOTHS:
    for R in RESOLUTION_MULTS:
        n_obs = sum(
            int((~torch.isnan(v[:, 2])).sum().item())
            for v in irr_maps[(ts, R)].values()
        )
        n_g   = next(iter(irr_maps[(ts, R)].values())).shape[0]
        print(f'  ts={ts}, R={R}: n_grid={n_g:6,}  n_valid={n_obs:8,}')

In [ ]:
# ── build thinned NNS map + spatial_coords per R ──────────────────────
# NNS must be recomputed on the thinned spatial positions.
# ordered_coords_1 is in (lat, lon) order — same convention as v1.

thinned_setups = {}

for R in RESOLUTION_MULTS:
    n_thin      = round(n_grid_full / R)
    thin_idx    = np.arange(0, n_thin)
    thin_coords = ordered_coords_1[thin_idx]   # (lat, lon)
    nns_thin    = _orderings.find_nns_l2(locs=thin_coords, max_nn=MM_COND_NUMBER)
    n_valid     = sum(
        int((~torch.isnan(v[:, 2])).sum().item())
        for v in irr_maps[(TRUE_SMOOTHS[0], R)].values()
    )
    thinned_setups[R] = {
        'thin_idx':       thin_idx,
        'spatial_coords': thin_coords,   # (lat, lon) for _build_shift_lookup
        'nns':            nns_thin,
        'n_grid':         n_thin,
        'n_obs_total':    n_valid,
    }
    print(f'R={R}: n_grid={n_thin:6,}  n_valid={n_valid:8,}  nns={nns_thin.shape}')

In [ ]:
# ── fitting loop: 16 fits ─────────────────────────────────────────────
results = []

for ts in TRUE_SMOOTHS:
    for R in RESOLUTION_MULTS:
        setup       = thinned_setups[R]
        irr_map_ord = irr_maps[(ts, R)]

        for fs in FIT_SMOOTHS:
            correctly_specified = (ts == fs)
            spec_label = 'CORRECT' if correctly_specified else 'MISSPEC'
            tag = f'true={ts}  fit={fs}  R={R}  [{spec_label}]'
            print(f"\n{'='*65}")
            print(f'Fitting: {tag}')
            print(f"  n_grid={setup['n_grid']}  n_obs={setup['n_obs_total']}")

            params = [
                torch.tensor([v], device=DEVICE, dtype=DTYPE, requires_grad=True)
                for v in init_log
            ]

            model = HybridVecchiaFit(
                smooth=fs,
                input_map={k: v.to(DEVICE) for k, v in irr_map_ord.items()},
                nns_map=setup['nns'],
                mm_cond_number=MM_COND_NUMBER,
                nheads=NHEADS,
                limit_A=LIMIT_A,
                limit_B_local=LIMIT_B_LOCAL,
                limit_C_local=LIMIT_C_LOCAL,
                daily_stride=DAILY_STRIDE,
                spatial_coords=setup['spatial_coords'],
                lag1_lon_offset=LAG1_LON_OFFSET,
                lag1_fresh_count=LAG1_FRESH,
                lag2_fresh_count=LAG2_FRESH,
            )
            model.precompute_conditioning_sets()

            optimizer = model.set_optimizer(
                params, lr=LBFGS_LR,
                max_iter=LBFGS_EVAL, max_eval=LBFGS_EVAL,
                history_size=LBFGS_HIST,
            )
            t0 = time.time()
            out, fit_iter = model.fit_vecc_lbfgs(
                params, optimizer, max_steps=LBFGS_STEPS, grad_tol=1e-5,
            )
            elapsed = time.time() - t0

            loss = float(out[-1])
            est  = backmap_params(out)
            print(f'  loss={loss:.4f}  elapsed={elapsed:.1f}s  steps={fit_iter+1}')
            for k in P_LABELS:
                re = abs(est[k] - TRUE_DICT[k]) / max(abs(TRUE_DICT[k]), 0.01)
                print(f'    {k:12s}: est={est[k]:8.4f}  true={TRUE_DICT[k]:8.4f}  re={re:.3f}')

            results.append({
                'true_smooth':         ts,
                'fit_smooth':          fs,
                'correctly_specified': correctly_specified,
                'R':                   R,
                'n_obs_total':         setup['n_obs_total'],
                'n_grid':              setup['n_grid'],
                'loss':                loss,
                'fit_steps':           fit_iter + 1,
                'elapsed_s':           round(elapsed, 1),
                **{f'est_{k}': est[k] for k in P_LABELS},
            })
            del model; gc.collect(); torch.cuda.empty_cache()

df = pd.DataFrame(results)
print(f'\nDone. {len(df)} fits recorded.')

In [ ]:
# ── parameter trajectory plots (identical visual encoding to v1) ───────
COLOR  = {0.5: 'steelblue', 1.5: 'tomato'}
LS     = {True: '-',        False: '--'}
LW     = {True: 2.5,        False: 1.5}
MK     = {True: 'o',        False: '^'}

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, (param, plabel) in enumerate(zip(P_LABELS, P_DISP)):
    ax = axes[i]
    tv = TRUE_DICT[param]

    for ts in TRUE_SMOOTHS:
        for fs in FIT_SMOOTHS:
            sub     = df[(df.true_smooth == ts) & (df.fit_smooth == fs)].sort_values('n_obs_total')
            correct = (ts == fs)
            ax.plot(
                sub['n_obs_total'], sub[f'est_{param}'],
                color=COLOR[ts], ls=LS[correct], lw=LW[correct],
                marker=MK[correct], ms=7,
                label=f'true={ts}, fit={fs}',
            )
            if correct and ts == 0.5:
                for _, row in sub.iterrows():
                    ax.annotate(
                        f"R={int(row.R)}",
                        (row['n_obs_total'], row[f'est_{param}']),
                        textcoords='offset points', xytext=(4, 4),
                        fontsize=7, color='grey',
                    )

    ax.axhline(tv, color='black', ls=':', lw=1.5, label=f'true={tv:.3g}')
    ax.set_title(plabel, fontsize=13)
    ax.set_xlabel('Valid obs count  (R=8 coarsest → R=1 finest)')
    ax.set_ylabel('Estimate')
    ax.legend(fontsize=7, loc='best')
    ax.grid(True, alpha=0.25)

axes[-1].set_visible(False)
legend_entries = [
    mlines.Line2D([], [], color=COLOR[0.5], ls='-',  lw=2.5, marker='o', label='true=0.5, fit=0.5 [correct]'),
    mlines.Line2D([], [], color=COLOR[0.5], ls='--', lw=1.5, marker='^', label='true=0.5, fit=1.5 [misspec]'),
    mlines.Line2D([], [], color=COLOR[1.5], ls='--', lw=1.5, marker='^', label='true=1.5, fit=0.5 [misspec]'),
    mlines.Line2D([], [], color=COLOR[1.5], ls='-',  lw=2.5, marker='o', label='true=1.5, fit=1.5 [correct]'),
    mlines.Line2D([], [], color='black',    ls=':',  lw=1.5,             label='true value'),
]
axes[-1].legend(handles=legend_entries, loc='center', fontsize=10, frameon=True)
axes[-1].set_visible(True)
axes[-1].axis('off')

fig.suptitle(
    'v2: MaxMin Thinning — Parameter Estimates vs Obs Count  (R=8 → R=1)\n'
    'Thick solid ● = correctly specified  |  Thin dashed ▲ = misspecified',
    fontsize=13, y=1.01,
)
plt.tight_layout()
plt.savefig('param_trajectories_v2.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: param_trajectories_v2.png')

In [ ]:
# ── RMSRE summary (identical to v1) ───────────────────────────────────
for k in P_LABELS:
    tv = TRUE_DICT[k]
    df[f're_{k}'] = (df[f'est_{k}'] - tv).abs() / max(abs(tv), 0.01)
re_cols = [f're_{k}' for k in P_LABELS]
df['RMSRE'] = df[re_cols].pow(2).mean(axis=1).pow(0.5)

print('=' * 70)
print('RESOLUTION SENSITIVITY v2 (MaxMin Thinning)')
print('=' * 70)
for ts in TRUE_SMOOTHS:
    for fs in FIT_SMOOTHS:
        spec = 'CORRECT' if ts == fs else 'MISSPEC'
        sub  = df[(df.true_smooth == ts) & (df.fit_smooth == fs)].sort_values('n_obs_total')
        print(f'\ntrue_smooth={ts}, fit_smooth={fs} [{spec}]')
        print(f"  {'n_obs':>8}  {'RMSRE':>8}  {'range_lon':>10}  {'nugget':>8}  {'advec_lon':>10}")
        for _, row in sub.iterrows():
            print(f"  {int(row.n_obs_total):>8}  {row.RMSRE:>8.4f}  "
                  f"{row.est_range_lon:>10.4f}  {row.est_nugget:>8.4f}  "
                  f"{row.est_advec_lon:>10.4f}")

print('\n' + '=' * 70)
print('ROBUSTNESS: RE range across all 16 fits per parameter')
print('=' * 70)
for k in P_LABELS:
    vals      = df[f're_{k}'].values
    mean_re   = vals.mean()
    range_re  = vals.max() - vals.min()
    print(f'  {k:12s}: mean_RE={mean_re:.4f}  range_RE={range_re:.4f}')